<center><h1>Keyword Spotting (KWS) with PyTorch</h1></center>

Train a small neural keyword spotter on Google Speech Commands v0.02 using **log-mel spectrogram** inputs and a **CNN** (plus an optional Depthwise-Separable CNN).

This notebook mirrors the dataset/split conventions used in `sklearn_keyword_spotting.ipynb` (official `validation_list.txt` / `testing_list.txt`).


<!--TABLE OF CONTENTS-->
# Table of Contents
- [Dataset: Speech Commands](#Dataset:-Speech-Commands)
- [Setup](#Setup)
- [Dataset Root (SPEECH_COMMANDS_DIR)](#Dataset-Root-(SPEECH_COMMANDS_DIR))
- [Split: train/val/test](#Split:-train/val/test)
- [Log-mel Features](#Log-mel-Features)
- [Dataset and DataLoaders](#Dataset-and-DataLoaders)
- [Model Architecture (PyTorch)](#Model-Architecture-(PyTorch))
- [Train](#Train)
- [Evaluate](#Evaluate)
- [Export](#Export)
- [Inference Example](#Inference-Example)


## Dataset: Speech Commands
This notebook expects the Speech Commands dataset already downloaded and extracted locally.

- Default path: `speech_commands_v0.02/`
- Override with env var: `SPEECH_COMMANDS_DIR=/path/to/speech_commands_v0.02`

The repository already includes `speech_commands_v0.02.tar.gz` and may include an extracted `speech_commands_v0.02/` folder.


## Setup


<!-- AUTO-DOC: 1 -->
### Imports and reproducibility
Goal
- Import libraries for audio loading, feature extraction, modeling, and metrics.
- Set a deterministic seed.

Notes
- `torch` is not listed in `requirements.txt` for the scikit-learn baseline. Install PyTorch separately if needed.


In [ ]:
from __future__ import annotations

import json
import os
import random
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable

import numpy as np
import matplotlib.pyplot as plt

import librosa

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torch.utils.data import Dataset, DataLoader
except Exception as e:
    raise ImportError(
        "PyTorch is required for this notebook. Install it first (https://pytorch.org/get-started/locally/) and restart the kernel.\n"
        f"Import error: {e}"
    )

from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

SEED = 0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DEVICE


<!-- AUTO-DOC: 2 -->
### Dataset root (SPEECH_COMMANDS_DIR)
Goal
- Locate your extracted Speech Commands dataset.
- (Optional) extract from the local `speech_commands_v0.02.tar.gz` if present.

Notes
- Downloading from the internet may be blocked in restricted environments; the repo includes a local archive.


In [ ]:
import tarfile

SPEECH_COMMANDS_DIR = Path(os.environ.get('SPEECH_COMMANDS_DIR', 'speech_commands_v0.02'))
LOCAL_ARCHIVE = Path('speech_commands_v0.02.tar.gz')

def _speech_commands_ok(root: Path) -> bool:
    return (root / 'validation_list.txt').exists() and (root / 'testing_list.txt').exists()

def _safe_extract(tar: tarfile.TarFile, path: Path) -> None:
    base = path.resolve()
    for member in tar.getmembers():
        member_path = (path / member.name).resolve()
        if base not in member_path.parents and member_path != base:
            raise RuntimeError(f'Unsafe path in tar file: {member.name}')
    tar.extractall(path)

if not _speech_commands_ok(SPEECH_COMMANDS_DIR):
    if LOCAL_ARCHIVE.exists():
        print(f'Extracting local archive: {LOCAL_ARCHIVE} -> {SPEECH_COMMANDS_DIR}')
        SPEECH_COMMANDS_DIR.mkdir(parents=True, exist_ok=True)
        with tarfile.open(LOCAL_ARCHIVE, 'r:gz') as tar:
            _safe_extract(tar, SPEECH_COMMANDS_DIR)

if not _speech_commands_ok(SPEECH_COMMANDS_DIR):
    raise FileNotFoundError(
        f"Expected Speech Commands dataset at: {SPEECH_COMMANDS_DIR.resolve()}\n"
        "Set SPEECH_COMMANDS_DIR to your extracted Speech Commands v0.02 folder.\n"
        "Expected files: validation_list.txt, testing_list.txt, and label subfolders with .wav files."
    )

SPEECH_COMMANDS_DIR


## Split: train/val/test
Speech Commands ships with `validation_list.txt` and `testing_list.txt`. We use those lists and treat everything else as training.


<!-- AUTO-DOC: 3 -->
### Official split lists
Goal
- Read the official split lists into fast lookup sets.


In [ ]:
def _read_list(path: Path) -> set[str]:
    lines = path.read_text(encoding='utf-8').splitlines()
    return {line.strip().replace(os.sep, '/') for line in lines if line.strip()}

val_list = _read_list(SPEECH_COMMANDS_DIR / 'validation_list.txt')
test_list = _read_list(SPEECH_COMMANDS_DIR / 'testing_list.txt')

len(val_list), len(test_list)


## Log-mel Features
Neural KWS models usually take a time-frequency representation (log-mel, MFCC, etc.) as input.

Here we use:
- 1 second clips at 16 kHz
- log-mel spectrogram with 40 mel bins


<!-- AUTO-DOC: 4 -->
### Audio and feature hyperparameters
Goal
- Define waveform and spectrogram settings.


In [ ]:
SAMPLE_RATE = 16_000
CLIP_SAMPLES = SAMPLE_RATE  # 1 second

N_FFT = 512
HOP_LENGTH = 160  # 10ms at 16kHz
N_MELS = 40
FMIN = 20
FMAX = SAMPLE_RATE // 2

# Numerical stability for log
LOG_EPS = 1e-6


<!-- AUTO-DOC: 5 -->
### Fixed-length audio loader and log-mel extraction
Goal
- Load audio as a 1s mono clip.
- Compute a log-mel spectrogram tensor shaped `[1, n_mels, n_frames]`.


In [ ]:
def load_audio_1s(path: Path, *, sample_rate: int = SAMPLE_RATE, clip_samples: int = CLIP_SAMPLES, rng: np.random.Generator | None = None) -> np.ndarray:
    y, _ = librosa.load(path, sr=sample_rate, mono=True)
    y = y.astype(np.float32)

    if len(y) < clip_samples:
        y = np.pad(y, (0, clip_samples - len(y)))
        return y

    if len(y) > clip_samples:
        max_offset = len(y) - clip_samples
        if rng is None:
            offset = max_offset // 2
        else:
            offset = int(rng.integers(0, max_offset + 1))
        y = y[offset : offset + clip_samples]

    return y


def log_mel_spectrogram(y: np.ndarray, *, sample_rate: int = SAMPLE_RATE) -> torch.Tensor:
    # mel: [n_mels, n_frames]
    mel = librosa.feature.melspectrogram(
        y=y,
        sr=sample_rate,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        n_mels=N_MELS,
        fmin=FMIN,
        fmax=FMAX,
        power=2.0,
    ).astype(np.float32)

    log_mel = np.log(mel + LOG_EPS)

    # Normalize per-example for stability
    log_mel = (log_mel - log_mel.mean()) / (log_mel.std() + 1e-6)

    # [1, n_mels, n_frames]
    return torch.from_numpy(log_mel).unsqueeze(0)


# Shape sanity check
x = log_mel_spectrogram(np.zeros(CLIP_SAMPLES, dtype=np.float32))
x.shape


## Dataset and DataLoaders
We mirror the baseline notebook’s label strategy:
- **Target words** (10 commands)
- **_unknown_** (all other words, balanced roughly to the number of training targets)
- **_silence_** (random 1s clips from `_background_noise_`)


<!-- AUTO-DOC: 6 -->
### Target labels and balancing knobs
Goal
- Choose a 10-keyword task and optional unknown/silence classes.


In [ ]:
TARGET_WORDS = [
    'yes', 'no', 'up', 'down', 'left', 'right', 'on', 'off', 'stop', 'go'
]

INCLUDE_SILENCE = True
UNKNOWN_LABEL = '_unknown_'
SILENCE_LABEL = '_silence_'

MAX_PER_LABEL = 1000  # None for full dataset (slower)


<!-- AUTO-DOC: 7 -->
### Discover labels in the dataset
Goal
- Enumerate label directories under the dataset root.


In [ ]:
def list_label_dirs(root: Path) -> list[str]:
    labels: list[str] = []
    for p in root.iterdir():
        if not p.is_dir():
            continue
        name = p.name
        if name.startswith('.'):
            continue
        if name in ['_background_noise_']:
            continue
        labels.append(name)
    return sorted(labels)

all_labels = list_label_dirs(SPEECH_COMMANDS_DIR)
missing = sorted(set(TARGET_WORDS) - set(all_labels))
if missing:
    raise ValueError(f"TARGET_WORDS not found under dataset root: {missing}")

len(all_labels), all_labels[:20]


<!-- AUTO-DOC: 8 -->
### Build Example records and assign splits
Goal
- Build a list of `(path, label, split)` examples using official split files.


In [ ]:
RNG = np.random.default_rng(SEED)


def iter_wavs(label: str) -> Iterable[Path]:
    return (SPEECH_COMMANDS_DIR / label).glob('*.wav')


def relpath_under_root(path: Path) -> str:
    try:
        return path.relative_to(SPEECH_COMMANDS_DIR).as_posix()
    except ValueError:
        return path.as_posix()


def split_of(rel: str) -> str:
    if rel in val_list:
        return 'val'
    if rel in test_list:
        return 'test'
    return 'train'


@dataclass(frozen=True)
class Example:
    path: Path
    label: str
    split: str


def build_examples(*, rng: np.random.Generator = RNG) -> list[Example]:
    examples: list[Example] = []

    # Target words
    for label in TARGET_WORDS:
        files = sorted(iter_wavs(label))
        if MAX_PER_LABEL is not None and len(files) > MAX_PER_LABEL:
            idx = rng.choice(len(files), size=MAX_PER_LABEL, replace=False)
            files = [files[i] for i in idx]

        for p in files:
            rel = relpath_under_root(p)
            examples.append(Example(path=p, label=label, split=split_of(rel)))

    # Unknown = everything else (balanced roughly to the number of training targets)
    unknown_labels = [l for l in all_labels if l not in TARGET_WORDS]
    unknown_files: list[Path] = []
    for l in unknown_labels:
        unknown_files.extend(list(iter_wavs(l)))
    unknown_files = sorted(unknown_files)

    target_train = sum(1 for e in examples if e.split == 'train')
    unknown_keep = min(len(unknown_files), target_train)
    if unknown_keep > 0:
        idx = rng.choice(len(unknown_files), size=unknown_keep, replace=False)
        for p in (unknown_files[i] for i in idx):
            rel = relpath_under_root(p)
            examples.append(Example(path=p, label=UNKNOWN_LABEL, split=split_of(rel)))

    # Silence samples from background noise (added to all splits)
    if INCLUDE_SILENCE:
        noise_dir = SPEECH_COMMANDS_DIR / '_background_noise_'
        if noise_dir.exists():
            noise_files = sorted(noise_dir.glob('*.wav'))
            if noise_files:
                silence_total = min(max(600, len(TARGET_WORDS) * 150), target_train)
                n_train = int(round(silence_total * 0.8))
                n_val = int(round(silence_total * 0.1))
                n_test = silence_total - n_train - n_val

                for split, n in [('train', n_train), ('val', n_val), ('test', n_test)]:
                    for _ in range(n):
                        p = noise_files[int(rng.integers(0, len(noise_files))) ]
                        examples.append(Example(path=p, label=SILENCE_LABEL, split=split))

    return examples


examples = build_examples()

from collections import Counter
Counter((e.split, e.label) for e in examples)


<!-- AUTO-DOC: 9 -->
### PyTorch Dataset
Goal
- Load a waveform, compute log-mel, and return `(features, class_index)`.

Notes
- For `_silence_`, we sample a random 1s window from a background-noise file.


In [ ]:
# Stable label ordering (important for metrics + export)
label_order = TARGET_WORDS + [UNKNOWN_LABEL]
if INCLUDE_SILENCE:
    label_order = [SILENCE_LABEL] + label_order

label_to_index = {lab: i for i, lab in enumerate(label_order)}


def _random_clip_from_noise(path: Path, *, rng: np.random.Generator) -> np.ndarray:
    y, _ = librosa.load(path, sr=SAMPLE_RATE, mono=True)
    y = y.astype(np.float32)

    if len(y) <= CLIP_SAMPLES:
        return np.pad(y, (0, max(0, CLIP_SAMPLES - len(y))))

    max_offset = len(y) - CLIP_SAMPLES
    offset = int(rng.integers(0, max_offset + 1))
    return y[offset : offset + CLIP_SAMPLES]


class KWSDataset(Dataset):
    def __init__(self, examples: list[Example], *, rng_seed: int = SEED):
        self.examples = examples
        self.rng = np.random.default_rng(rng_seed)

    def __len__(self) -> int:
        return len(self.examples)

    def __getitem__(self, idx: int):
        ex = self.examples[idx]
        if ex.label == SILENCE_LABEL:
            y = _random_clip_from_noise(ex.path, rng=self.rng)
        else:
            y = load_audio_1s(ex.path, rng=self.rng)

        x = log_mel_spectrogram(y)
        y_idx = label_to_index[ex.label]
        return x, y_idx


def select_split(split: str) -> list[Example]:
    return [e for e in examples if e.split == split]

train_ex = select_split('train')
val_ex = select_split('val')
test_ex = select_split('test')

len(train_ex), len(val_ex), len(test_ex), label_order


<!-- AUTO-DOC: 10 -->
### DataLoaders
Goal
- Build DataLoaders for train/val/test.


In [ ]:
BATCH_SIZE = 64
NUM_WORKERS = 0  # increase if your environment supports it

train_ds = KWSDataset(train_ex, rng_seed=SEED)
val_ds = KWSDataset(val_ex, rng_seed=SEED + 1)
test_ds = KWSDataset(test_ex, rng_seed=SEED + 2)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=(DEVICE.type=='cuda'))
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=(DEVICE.type=='cuda'))
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=(DEVICE.type=='cuda'))

# Inspect one batch
xb, yb = next(iter(train_loader))
xb.shape, yb.shape, xb.dtype


## Model Architecture (PyTorch)
We define two architectures you can swap between:

1. `KWSConvNet`: a simple Conv/BN/ReLU stack.
2. `KWSDscnn`: depthwise-separable conv blocks (common in small-footprint KWS).

Both consume an input shaped `[B, 1, n_mels, n_frames]`.


<!-- AUTO-DOC: 11 -->
### Model definitions
Goal
- Implement compact CNN architectures for keyword classification.


In [ ]:
def count_params(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


class KWSConvNet(nn.Module):
    def __init__(self, num_classes: int):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2),

            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2),

            nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(128, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.features(x)
        return self.classifier(x)


class DepthwiseSeparableConv2d(nn.Module):
    def __init__(self, in_ch: int, out_ch: int, *, kernel_size: int = 3, stride: int = 1, padding: int = 1):
        super().__init__()
        self.depthwise = nn.Conv2d(in_ch, in_ch, kernel_size=kernel_size, stride=stride, padding=padding, groups=in_ch, bias=False)
        self.pointwise = nn.Conv2d(in_ch, out_ch, kernel_size=1, bias=False)
        self.bn = nn.BatchNorm2d(out_ch)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.depthwise(x)
        x = self.pointwise(x)
        x = self.bn(x)
        return F.relu(x, inplace=True)


class KWSDscnn(nn.Module):
    def __init__(self, num_classes: int):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(1, 64, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
        )
        self.blocks = nn.Sequential(
            DepthwiseSeparableConv2d(64, 64),
            nn.MaxPool2d(kernel_size=2),
            DepthwiseSeparableConv2d(64, 128),
            nn.MaxPool2d(kernel_size=2),
            DepthwiseSeparableConv2d(128, 128),
        )
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(128, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.stem(x)
        x = self.blocks(x)
        return self.head(x)


NUM_CLASSES = len(label_order)
model = KWSDscnn(NUM_CLASSES).to(DEVICE)  # swap to KWSConvNet(NUM_CLASSES) if you prefer

print(model)
print('trainable params:', count_params(model))


## Train


<!-- AUTO-DOC: 12 -->
### Training utilities
Goal
- Implement `train_one_epoch` and `evaluate` loops.


In [ ]:
def accuracy(logits: torch.Tensor, targets: torch.Tensor) -> float:
    preds = logits.argmax(dim=1)
    return (preds == targets).float().mean().item()


@torch.no_grad()
def evaluate(model: nn.Module, loader: DataLoader) -> dict:
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total = 0

    for xb, yb in loader:
        xb = xb.to(DEVICE)
        yb = yb.to(DEVICE)

        logits = model(xb)
        loss = F.cross_entropy(logits, yb)

        total_loss += loss.item() * yb.size(0)
        total_correct += (logits.argmax(dim=1) == yb).sum().item()
        total += yb.size(0)

    return {
        'loss': total_loss / max(1, total),
        'acc': total_correct / max(1, total),
    }


def train_one_epoch(model: nn.Module, loader: DataLoader, optimizer: torch.optim.Optimizer) -> dict:
    model.train()
    total_loss = 0.0
    total_correct = 0
    total = 0

    for xb, yb in loader:
        xb = xb.to(DEVICE)
        yb = yb.to(DEVICE)

        optimizer.zero_grad(set_to_none=True)
        logits = model(xb)
        loss = F.cross_entropy(logits, yb)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * yb.size(0)
        total_correct += (logits.argmax(dim=1) == yb).sum().item()
        total += yb.size(0)

    return {
        'loss': total_loss / max(1, total),
        'acc': total_correct / max(1, total),
    }


<!-- AUTO-DOC: 13 -->
### Run training
Goal
- Train for a few epochs and track validation accuracy.

Tips
- Start with a small `MAX_PER_LABEL` while iterating.
- If you have a GPU, training is much faster.


In [ ]:
EPOCHS = 5
LR = 1e-3
WEIGHT_DECAY = 1e-4

optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

history = []
for epoch in range(1, EPOCHS + 1):
    tr = train_one_epoch(model, train_loader, optimizer)
    va = evaluate(model, val_loader)
    row = {"epoch": epoch, **{f"train_{k}": v for k, v in tr.items()}, **{f"val_{k}": v for k, v in va.items()}}
    history.append(row)
    print(row)

history[-1]


## Evaluate


<!-- AUTO-DOC: 14 -->
### Test-set report + confusion matrix
Goal
- Compute a classification report and confusion matrix on the held-out test split.


In [ ]:
@torch.no_grad()
def predict_all(model: nn.Module, loader: DataLoader) -> tuple[np.ndarray, np.ndarray]:
    model.eval()
    y_true = []
    y_pred = []

    for xb, yb in loader:
        xb = xb.to(DEVICE)
        logits = model(xb)
        preds = logits.argmax(dim=1).cpu().numpy()
        y_pred.append(preds)
        y_true.append(yb.numpy())

    return np.concatenate(y_true), np.concatenate(y_pred)


y_true, y_pred = predict_all(model, test_loader)

print(classification_report(y_true, y_pred, target_names=label_order, zero_division=0))

cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(10, 10))
ConfusionMatrixDisplay(cm, display_labels=label_order).plot(ax=ax, xticks_rotation=45, cmap='Blues', colorbar=False)
ax.set_title('Keyword Spotting Confusion Matrix (PyTorch)')
plt.show()


## Export
We export:
- `pytorch_kws_model.pt`: `state_dict` + small metadata
- `pytorch_kws_labels.json`: label ordering used for training


<!-- AUTO-DOC: 15 -->
### Save model and labels
Goal
- Save the trained model weights and label list for later inference.


In [ ]:
OUTPUT_DIR = Path('output')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = OUTPUT_DIR / 'pytorch_kws_model.pt'
LABELS_PATH = OUTPUT_DIR / 'pytorch_kws_labels.json'

payload = {
    'model_type': model.__class__.__name__,
    'state_dict': model.state_dict(),
    'sample_rate': SAMPLE_RATE,
    'clip_samples': CLIP_SAMPLES,
    'n_fft': N_FFT,
    'hop_length': HOP_LENGTH,
    'n_mels': N_MELS,
    'fmin': FMIN,
    'fmax': FMAX,
}

torch.save(payload, MODEL_PATH)
LABELS_PATH.write_text(json.dumps(label_order, indent=2), encoding='utf-8')

MODEL_PATH, LABELS_PATH


## Inference Example
Run the exported model on a single `.wav` file.


<!-- AUTO-DOC: 16 -->
### Load + predict
Goal
- Load the saved weights.
- Run a forward pass on one file.


In [ ]:
# Pick any WAV under the dataset (or set your own path)
example_wav = next((SPEECH_COMMANDS_DIR / TARGET_WORDS[0]).glob('*.wav'))
example_wav


In [ ]:
labels = json.loads(LABELS_PATH.read_text(encoding='utf-8'))
loaded = torch.load(MODEL_PATH, map_location=DEVICE)

# Recreate the model (match MODEL_TYPE)
if loaded['model_type'] == 'KWSConvNet':
    infer_model = KWSConvNet(len(labels))
elif loaded['model_type'] == 'KWSDscnn':
    infer_model = KWSDscnn(len(labels))
else:
    raise ValueError(f"Unknown model_type: {loaded['model_type']}")

infer_model.load_state_dict(loaded['state_dict'])
infer_model.to(DEVICE)
infer_model.eval()

# Feature + predict
rng = np.random.default_rng(SEED)
y = load_audio_1s(example_wav, rng=rng)
x = log_mel_spectrogram(y).unsqueeze(0).to(DEVICE)  # [1, 1, n_mels, n_frames]

with torch.no_grad():
    logits = infer_model(x)
    probs = logits.softmax(dim=1).squeeze(0).cpu().numpy()

pred_idx = int(probs.argmax())
print('file:', example_wav)
print('pred:', labels[pred_idx])
print('top5:')
for i in probs.argsort()[-5:][::-1]:
    print(f"  {labels[int(i)]:>10s}: {probs[int(i)]:.3f}")
